In [ ]:
%pip -q install requests google-api-python-client google-auth
import hashlib
import json
from urllib.request import Request, urlopen

REPO = "joao9vegeto-stack/KintTany"
BRANCH = "KaKaRoTiOS"
SCRIPT_PATH = "colab/KaKaRoTiOS_GitHub_to_Drive.py"
EXPECTED_SCRIPT_BLOB_SHA = "31f27e729972aa194d051d37bc62b5f965f2d628"

# O SHA do blob fixa exatamente o script esperado, mesmo usando a branch como URL.
contents_url = f"https://api.github.com/repos/{REPO}/contents/{SCRIPT_PATH}?ref={BRANCH}"
metadata_request = Request(
    contents_url,
    headers={"Accept": "application/vnd.github+json", "User-Agent": "KaKaRoTiOS-Colab"},
)
with urlopen(metadata_request, timeout=45) as response:
    metadata = json.load(response)
if metadata.get("sha") != EXPECTED_SCRIPT_BLOB_SHA:
    raise RuntimeError(
        "Script remoto não corresponde ao SHA fixado neste notebook; nada será executado. "
        f"Esperado={EXPECTED_SCRIPT_BLOB_SHA}; recebido={metadata.get('sha')}"
    )

SCRIPT_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/{SCRIPT_PATH}"
with urlopen(SCRIPT_URL, timeout=90) as response:
    source_bytes = response.read()
actual_blob_sha = hashlib.sha1(
    f"blob {len(source_bytes)}\\0".encode("ascii") + source_bytes
).hexdigest()
if actual_blob_sha != EXPECTED_SCRIPT_BLOB_SHA:
    raise RuntimeError(
        "O conteúdo baixado mudou durante a leitura ou não corresponde ao SHA fixado; nada será executado. "
        f"Esperado={EXPECTED_SCRIPT_BLOB_SHA}; recebido={actual_blob_sha}"
    )
source = source_bytes.decode("utf-8")
if len(source) < 20000 or "UPLOAD_VERIFICADO — SUCESSO" not in source:
    raise RuntimeError("Script remoto incompleto ou inesperado; nada será executado.")
if "KaKaRoTiOS-Kakarot-" not in source or "ARTIFACT_NAME_PREFIX" not in source:
    raise RuntimeError("Script remoto não contém o seletor de artefatos Kakarot atualizado; nada será executado.")
if "010051C0134F8000" not in source:
    raise RuntimeError("Script remoto não corresponde ao Title ID do Kakarot; nada será executado.")
compile(source, "KaKaRoTiOS_GitHub_to_Drive.py", "exec")
print(f"Script validado por Git blob SHA: {actual_blob_sha}; iniciando transferência KaKaRoTiOS.")
exec(compile(source, "KaKaRoTiOS_GitHub_to_Drive.py", "exec"))
